# Yfinance API tutorial
In this tutorial, we will learn how to use the `yfinance` library to fetch financial data from Yahoo Finance. The `yfinance` library allows you to easily access historical market data, stock information, and financial statements. 

Focus of Ticker.info and Ticker.fast_info for common asset types

In [4]:
import yfinance as yf

# to get memory size of objects
from pympler.asizeof import asizeof

# simple way to use/test rich capacities (no rich.console)
# rich included in yfinance (to check when it is used )
# without renaming replace the usual print function
from rich import print as rprint
from rich import print_json

from rich.console import Console

console = Console()

import textwrap

# activate debug
# yf.config.debug.logging = True
# set to False to stop yfinance hiding exceptions.
# yf.config.debug.hide_exceptions = False

# show yfinance configuration
rprint(yf.config)  # some output (<..>) are badly rendered
# console.print(yf.config, highlight=False) # highlight False correct the defaut


{
    "network": {
        "proxy": null,
        "retries": 0
    },
    "debug": {
        "hide_exceptions": true,
        "logging": false
    },
    "locale": {
        "lang": "en-US",
        "region": "US"
    },
    "__rich__": {},
    "aihwerij235234ljsdnp34ksodfipwoe234234jlskjdf": {}
}

## Architecture

### Storage of tickers in a container, yahoo_code, isin, bourso_code

**AssetsTicker**

It is necessary to retrieve financial data for ALL assets

- simple dictionary of ticker : contains all stocks, forex, indices...
    - simpler all tickers have a uuid (ticker), or tuple (ticker, MIC) - yahoo finance specific
    - add a type enumeration (**AssetType**) to know which type of data are present
        - can we retrive from the comman ticker ?

Responsability : provide tickers to callers
    - one or more callers ? potentially a boursorama parser, UI information

### Rerieve financial data

From the different assets, specific data are accessible:
    - An *equity* provides dividends, publish *financial* (`balance_sheet`, `quaterly_income_stmt`). May have associated *derivatives*
    - A *fund* or *mutual_fund* have data related to the composition `Ticker::FundsData*`
- More organized different class for each type
    1. base class **Asset** and child **Equity**, **Forex**
    2. the detail of assetType is maybe too detailed (for the use here), 
        we simply have some categories of data, implement like a mix-in (Dividends, Shares) 


In [9]:
from enum import Enum


class AssetType(Enum):
    """
    Different types of Assets
    """

    UNDEFINED = 0
    EQUITY = 1  # Action
    forex = 2  # Indices
    FOREX = 3  # Devise
    DIGITAL_ASSET = 4  # all kinds crypto(BTC, ETH), NFT, Stablecoins...
    MUTUAL_FUND = 5  # OPCVM, content available by FundsData
    COMMODITY = 6  # Matières premières
    # Not sure all those distinctions are important for Yahoo Finance retrieval data
    FUTURE = 7  #
    DERIVATIVES = 8  # OPTION, WARRANT, CFD
    ETF = 9  # ETF on various assets
    RATES = 10  # Taux institutionnels (non accessible aux particuliers)
    BOND = 11  # (Obligation) PRIVATE_BOND and GOVERNMENT_BOND
    MONEY_MARKET = 12  # Taux à court termes


# maiki.ticker classs for one ticker
dict_tickers = {
    "microsoft": {
        "yfTicker": "MSFT",
        "yfIsin": "US5949181045",
        "type": AssetType.EQUITY,
    },
    "quantum": {"yfTicker": "QNT", "yfIsin": "US7479066000", "type": AssetType.EQUITY},
    "carrefour": {"yfTicker": "CA.PA", "type": AssetType.EQUITY},
    "cac40": {
        "yfTicker": "^FCHI",
        "yfIsin": "FR0003500008",
        "type": AssetType.forex,
    },
    "eurusd": {"yfTicker": "EURUSD=X", "yfIsin": "", "type": AssetType.FOREX},
    "natixis_horizon_40_44": {
        "yfTicker": "0P00014IGT.F",
        "yfIsin": "FR0011461276",
        "type": AssetType.MUTUAL_FUND,
    },
}

# rprint(dict_tickers)
console.print(dict_tickers, highlight=False, overflow="fold")

{
    'microsoft': {'yfTicker': 'MSFT', 'yfIsin': 'US5949181045', 'type': <AssetType.EQUITY: 1>},
    'quantum': {'yfTicker': 'QNT', 'yfIsin': 'US7479066000', 'type': <AssetType.EQUITY: 1>},
    'carrefour': {'yfTicker': 'CA.PA', 'type': <AssetType.EQUITY: 1>},
    'cac40': {'yfTicker': '^FCHI', 'yfIsin': 'FR0003500008', 'type': <AssetType.forex: 2>},
    'eurusd': {'yfTicker': 'EURUSD=X', 'yfIsin': '', 'type': <AssetType.FOREX: 3>},
    'natixis_horizon_40_44': {
        'yfTicker': '0P00014IGT.F',
        'yfIsin': 'FR0011461276',
        'type': <AssetType.MUTUAL_FUND: 5>
    }
}

## Asset

Common to all:
    - `fast_info` is hardcoded, always 20 keys

## Equity
Real entreprises:
- BusinessSummary 
- Sector,Industry
- dividends
- eps

In [ ]:
def print_generic_asset(asset: yf.Ticker):
    print("\n== generic asset ==")
    print(f"ticker: {asset.ticker}")
    print(f"Isin: {asset.isin}")

    # FastInfo
    finfo = asset.fast_info
    print(f"fast info nb keys: {len(finfo.keys())}")
    print(
        f"finfo quoteType({type(finfo['quoteType'])}):", finfo["quoteType"]
    )  # => EQUITY
    print(f"finfo currency({type(finfo['currency'])}): ", finfo["currency"])  # USD
    print(
        f"finfo exchange({type(finfo['exchange'])}):", finfo["exchange"]
    )  # => NMS (need full_info for full name Nasdasq...)
    print(
        f"finfo timezone({type(finfo['timezone'])}):", finfo["timezone"]
    )  # => America/New_York
    print(f"finfo open({type(finfo['open'])}): ", finfo["open"])
    print(f"finfo last_price({type(finfo['last_price'])}): ", finfo["last_price"])

    # NONE for AssetType.INDEX, FOREX
    # valuable for EQUITY
    print(
        f"finfo marketCap({type(finfo['marketCap'])})", finfo["marketCap"]
    )  # (<class 'float'>) 3095205987862.728
    print(f"finfo shares({type(finfo['shares'])}): ", finfo["shares"])

    # Info
    info = asset.info
    print(f"info nb keys: {len(info)}")
    print(
        f"info marketState({type(info['marketState'])}):", info["marketState"]
    )  # REGULAR / POSTPOST
    print(
        f"info tradeable({type(info['tradeable'])}):", info["tradeable"]
    )  # Boolean False

    # It is a compound of many objects, lazy loaded
    print(f"Size object: {asizeof(asset)} bytes")


# interesting attributes
def print_common(stock: yf.Ticker):
    # base class generic info
    print_generic_asset(stock)

    # print("\n == Equity ==")
    console.rule("[bold magenta] Common", style="blue")

    finfo = stock.fast_info

    # dividends : DataFrames
    print(f"Dividends ({type(stock.dividends)}) : {stock.dividends.tail(1)}")

    print(
        f"finfo marketCap({type(finfo['marketCap'])})", finfo["marketCap"]
    )  # (<class 'float'>) 3095205987862.728
    print(f"finfo shares: ", finfo["shares"])
    print(f"finfo open({type(finfo['open'])}): ", finfo["open"])
    print(
        f"finfo last_price({type(finfo['last_price'])}): ", finfo["last_price"]
    )  # up-to-date apriori

    # It is a compound of many objects, lazy loaded
    print(f"Size object: {asizeof(stock)} bytes")


# longBusinessSummary creates error for most, except EQUITY
def print_equity(stock: yf.Ticker):
    # base class generic info
    # print_generic_asset(stock)

    print_common(stock)

    businessSummary: str = textwrap.shorten(
        stock.info["longBusinessSummary"], width=25, placeholder="..."
    )
    rprint(f"businessSummary (str) : {businessSummary}")
    print(f"Sector: {stock.info['sector']}")
    print(f"Industry: {stock.info['industry']}")


In [ ]:
ticker = dict_tickers["microsoft"]

# stock = yf.Ticker(ticker["yfTicker"])
stock = yf.Ticker(ticker["yfIsin"])

print_equity(stock)
# textwrap.shorten(stock.info["longBusinessSummary"], width=25, placeholder="...")
# fetch Quote.info return dict
# print(stock.info)
# new fetch => Series
# stock.dividends
# stock.earnings_history
# stock.eps_trend
# stock.calendar
# stock.fast_info.exchange

## Index

Not present:
- info / longBusinessSumary : key not present
- finfo / marketCap : None
- finfo / shares : None

Error:
- no info / sector or  industry

In [ ]:
ticker_index = dict_tickers["cac40"]

index = yf.Ticker(ticker_index["yfTicker"])
# print_equity(index)
# fields of info are different: longBusinessSummary not present
# print( f"info: ", index.info)
print(f"info nb keys: {len(index.info)}")
rprint(f"info: ", index.info)
print(f"info nb keys: {len(index.fast_info.keys())}")
rprint(f"fast_info: {index.fast_info}")

# print_generic_asset(index)
print_common(index)


## FOREX

finfo / quoteType: FOREX
finfo / exchange CCY
info / marKetState Regular
info / tradeable False

finfo marketCap / shares : None

In [ ]:
## FOREX
forex_index = dict_tickers["eurusd"]

forex = yf.Ticker(forex_index["yfTicker"])

print(f"info nb keys: {len(forex.info)}")
rprint(f"info: ", forex.info)
print(f"info nb keys: {len(forex.fast_info.keys())}")
rprint(f"fast_info: {forex.fast_info}")

console.rule("[bold] JSON Fast Info")
json_str = forex.fast_info.toJSON()
rprint(json_str)
# console.rule("[bold magenta] JSON Fast Info")
# print_json(json_str)

# print_generic_asset(forex)
print_common(forex)


## OPCVM
- finfo quoteType MUTUALFUND
- finfo / last_price is ok
- open is None
- no marketCap, shares
- marketState REGULAR, tradeable False

error:
- longBusinessSummary

In [ ]:
opcvm = yf.Ticker("FR0011461276")

# print(f"info nb keys: {len(opcvm.info)}")
# rprint(f"info: ", opcvm.info)
# print(f"info nb keys: {len(opcvm.fast_info.keys())}")
# rprint(f"fast_info: {opcvm.fast_info}")

print_common(opcvm)


rprint(opcvm.financials)  # Empty DataFrame
# rprint(opcvm.funds_data) # <yfinance.scrapers.funds.FundsData object at 0x7fe6a7407b60>
fundData = opcvm.funds_data
# rprint(f"FundsData/asset_classes: {fundData.asset_classes}")
rprint(f"FundsData/asset_classes: ", fundData.asset_classes)
print(f"FundsData/bond_holdings: ", fundData.bond_holdings)  # DataFrame
rprint(f"FundsData/description: ", fundData.description)
print(f"FundsData/equity_holdings: ", fundData.equity_holdings)
rprint(f"FundsData/sector_weightings: ", fundData.sector_weightings)
rprint(f"FundsData/fund_overview: ", fundData.fund_overview)

# maybe others

In [ ]:
## History

In [ ]:
print(opcvm.ticker)
# print(opcvm.calendar)

# opcvm.history
data = opcvm.history(
    period="1mo",
)
print(type(data))
print(f"data shape: {data.shape}")
print(f"data columns: {data.columns}")
print(data)

# print(opcvm.history_metadata)